# TA-004G — GradCAM: EfficientNet-B0 vs DenseNet-121

Visualización de activaciones GradCAM para ambos modelos reentrenados (TA-004E y TA-004F).  
Objetivo: entender qué regiones de la radiografía activa cada modelo por clase.

## 1. Instalación

In [ ]:
#%pip install grad-cam -q
print('grad-cam OK')

Note: you may need to restart the kernel to use updated packages.
grad-cam OK



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [2]:
import json
import numpy as np
import pandas as pd
import pydicom
import wandb
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from torchvision import models, transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


## 3. Configuración

In [3]:
BASE       = Path(r'E:\Taller Integrador I\ModelosIA\Radiografias')
TEST_DIR   = BASE / 'dataset_split' / 'test'
MANIFESTS  = BASE / 'dataset_split' / 'manifests'
NB_DIR     = Path(r'E:\Taller Integrador I\ModelosIA\modelos\Notebooks')
CKPT_DIR   = Path(r'E:\Taller Integrador I\ModelosIA\modelos\checkpoints')

CLASSES     = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']
NUM_CLASSES = len(CLASSES)
IMG_SIZE    = 224
N_SAMPLES   = 3

with open(NB_DIR / 'best_params_efficientnet.json') as f:
    eff_params = json.load(f)
with open(NB_DIR / 'best_params_densenet.json') as f:
    dn_params = json.load(f)

DROPOUT_EFF = eff_params['params']['dropout']
DROPOUT_DN  = dn_params['params']['dropout']

print(f'EfficientNet dropout : {DROPOUT_EFF:.4f}')
print(f'DenseNet dropout     : {DROPOUT_DN:.4f}')
print(f'N samples per class  : {N_SAMPLES}')

EfficientNet dropout : 0.4832
DenseNet dropout     : 0.2739
N samples per class  : 3


## 4. W&B Init

In [4]:
wandb.init(
    project='vetxray-cnn',
    entity='dbaylont1-antenor-orrego-private-university',
    name='TA-004G-GradCAM',
    config={
        'task':              'TA-004G',
        'models':            ['efficientnet_b0', 'densenet121'],
        'n_samples_per_class': N_SAMPLES,
        'method':            'GradCAM',
        'eff_checkpoint':    'best_efficientnet_v2.pth',
        'dn_checkpoint':     'best_densenet_v2.pth'
    },
    tags=['gradcam', 'visualization', 'TA-004G', 'VetXRay']
)
print('W&B run:', wandb.run.url)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Pcuser\_netrc.
wandb: Currently logged in as: dbaylont1 (dbaylont1-antenor-orrego-private-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/dbaylont1-antenor-orrego-private-university/vetxray-cnn/runs/dj20ew6k


## 5. Utilidades DICOM

In [6]:
TRANSFORM_INF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def load_dicom_for_gradcam(path):
    dcm = pydicom.dcmread(str(path))
    arr = dcm.pixel_array.astype(np.float32)
    if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == 'MONOCHROME1':
        arr = arr.max() - arr
    p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
    arr = np.clip(arr, p2, p98)
    arr = (arr - p2) / (p98 - p2 + 1e-8)
    img_gray = Image.fromarray((arr * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    img_rgb   = Image.merge('RGB', [img_gray, img_gray, img_gray])
    rgb_float = np.array(img_rgb, dtype=np.float32) / 255.0
    tensor    = TRANSFORM_INF(img_rgb).unsqueeze(0).to(DEVICE)
    return rgb_float, tensor

def get_prob(model, tensor, cls_idx):
    with torch.no_grad():
        return torch.sigmoid(model(tensor))[0, cls_idx].item()

print('Utilidades DICOM OK')

Utilidades DICOM OK


## 6. Selección de muestras del test set

In [7]:
df_test = pd.read_csv(MANIFESTS / 'test.csv')
print(f'Test total: {len(df_test)} imágenes')

samples = {}
for cls in CLASSES:
    mask   = df_test['TAG'].str.contains(cls, na=False)
    cls_df = df_test[mask].reset_index(drop=True)
    samples[cls] = cls_df.head(N_SAMPLES).to_dict('records')
    print(f'  {cls:<22}: {mask.sum():3d} positivos — usando {len(samples[cls])} para GradCAM')

Test total: 940 imágenes
  alveolar_pattern      : 127 positivos — usando 3 para GradCAM
  bronchial_pattern     :  72 positivos — usando 3 para GradCAM
  pleural_effusion      :  61 positivos — usando 3 para GradCAM
  cardiomegaly          : 238 positivos — usando 3 para GradCAM
  no_finding            : 453 positivos — usando 3 para GradCAM


## 7. Carga de modelos

In [8]:
def build_efficientnet(dropout):
    m = models.efficientnet_b0(weights=None)
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, NUM_CLASSES))
    return m

def build_densenet(dropout):
    m = models.densenet121(weights=None, memory_efficient=False)
    in_f = m.classifier.in_features
    m.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, NUM_CLASSES))
    return m

eff_model = build_efficientnet(DROPOUT_EFF)
eff_model.load_state_dict(torch.load(CKPT_DIR / 'best_efficientnet_v2.pth', map_location=DEVICE))
eff_model = eff_model.eval().to(DEVICE)
print('EfficientNet-B0 cargado OK')

dn_model = build_densenet(DROPOUT_DN)
dn_model.load_state_dict(torch.load(CKPT_DIR / 'best_densenet_v2.pth', map_location=DEVICE))
dn_model = dn_model.eval().to(DEVICE)
print('DenseNet-121 cargado OK')

print(f'\nEfficientNet classifier: {eff_model.classifier}')
print(f'DenseNet classifier    : {dn_model.classifier}')

EfficientNet-B0 cargado OK
DenseNet-121 cargado OK

EfficientNet classifier: Sequential(
  (0): Dropout(p=0.4832290311184182, inplace=False)
  (1): Linear(in_features=1280, out_features=5, bias=True)
)
DenseNet classifier    : Sequential(
  (0): Dropout(p=0.2739417822102108, inplace=False)
  (1): Linear(in_features=1024, out_features=5, bias=True)
)


## 8. Configuración GradCAM

In [9]:
eff_target_layer = eff_model.features[-1]
dn_target_layer  = dn_model.features.norm5

cam_eff = GradCAM(model=eff_model, target_layers=[eff_target_layer])
cam_dn  = GradCAM(model=dn_model,  target_layers=[dn_target_layer])

print(f'EfficientNet target layer: features[-1] → {eff_target_layer.__class__.__name__}')
print(f'DenseNet target layer    : features.norm5 → {dn_target_layer.__class__.__name__}')

EfficientNet target layer: features[-1] → Conv2dNormActivation
DenseNet target layer    : features.norm5 → BatchNorm2d


## 9. Función GradCAM grid

In [10]:
CLASS_LABELS = {
    'alveolar_pattern':  'Patrón\nAlveolar',
    'bronchial_pattern': 'Patrón\nBronquial',
    'pleural_effusion':  'Derrame\nPleural',
    'cardiomegaly':      'Cardio-\nmegalia',
    'no_finding':        'Sin\nHallazgo'
}

def generate_gradcam_grid(cam_obj, model, model_name, samples_dict, save_path):
    n_rows = len(CLASSES)
    n_cols = N_SAMPLES
    fig, axes = plt.subplots(n_rows, n_cols + 1, figsize=(4.5*(n_cols+1), 4.5*n_rows))
    fig.suptitle(f'GradCAM — {model_name}', fontsize=16, fontweight='bold', y=1.01)

    for j in range(n_cols):
        axes[0, j+1].set_title(f'Muestra {j+1}', fontsize=10, fontweight='bold')

    for i, cls in enumerate(CLASSES):
        axes[i, 0].text(0.5, 0.5, CLASS_LABELS[cls],
                        ha='center', va='center', fontsize=11, fontweight='bold',
                        transform=axes[i, 0].transAxes)
        axes[i, 0].axis('off')

        cls_idx = i
        records = samples_dict.get(cls, [])

        for j, rec in enumerate(records[:n_cols]):
            ax = axes[i, j+1]
            try:
                img_path = TEST_DIR / rec['FileName']
                rgb_float, tensor = load_dicom_for_gradcam(img_path)
                grayscale_cam = cam_obj(input_tensor=tensor,
                                        targets=[ClassifierOutputTarget(cls_idx)])
                overlay = show_cam_on_image(rgb_float, grayscale_cam[0], use_rgb=True)
                prob    = get_prob(model, tensor, cls_idx)
                ax.imshow(overlay)
                color = 'green' if prob >= 0.5 else 'red'
                ax.set_title(f'P = {prob:.3f}', fontsize=9, color=color)
            except Exception as e:
                ax.text(0.5, 0.5, f'Error:\n{str(e)[:50]}',
                        ha='center', va='center', fontsize=7, transform=ax.transAxes, color='red')
            ax.axis('off')

        for j in range(len(records), n_cols):
            axes[i, j+1].axis('off')

    plt.tight_layout()
    plt.savefig(str(save_path), dpi=100, bbox_inches='tight')
    print(f'Guardado: {save_path.name}')
    plt.show()
    return fig

print('generate_gradcam_grid OK')

generate_gradcam_grid OK


## 10. GradCAM — EfficientNet-B0

In [11]:
path_eff = NB_DIR / 'ta004g_gradcam_efficientnet.png'
fig_eff  = generate_gradcam_grid(cam_eff, eff_model, 'EfficientNet-B0 (TA-004E)', samples, path_eff)
wandb.log({'GradCAM/EfficientNet': wandb.Image(str(path_eff))})

Guardado: ta004g_gradcam_efficientnet.png


C:\Users\Pcuser\AppData\Local\Temp\ipykernel_14936\1442794638.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. GradCAM — DenseNet-121

In [12]:
path_dn = NB_DIR / 'ta004g_gradcam_densenet.png'
fig_dn  = generate_gradcam_grid(cam_dn, dn_model, 'DenseNet-121 (TA-004F)', samples, path_dn)
wandb.log({'GradCAM/DenseNet': wandb.Image(str(path_dn))})

Guardado: ta004g_gradcam_densenet.png


C:\Users\Pcuser\AppData\Local\Temp\ipykernel_14936\1442794638.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Comparación EfficientNet vs DenseNet (1 muestra por clase)

In [13]:
fig_cmp, axes = plt.subplots(len(CLASSES), 3, figsize=(13, 5*len(CLASSES)))
fig_cmp.suptitle('GradCAM — Comparación EfficientNet vs DenseNet', fontsize=14, fontweight='bold')

axes[0, 0].set_title('Clase', fontsize=10, fontweight='bold')
axes[0, 1].set_title('EfficientNet-B0', fontsize=10, fontweight='bold')
axes[0, 2].set_title('DenseNet-121', fontsize=10, fontweight='bold')

for i, cls in enumerate(CLASSES):
    axes[i, 0].text(0.5, 0.5, CLASS_LABELS[cls],
                    ha='center', va='center', fontsize=12, fontweight='bold',
                    transform=axes[i, 0].transAxes)
    axes[i, 0].axis('off')

    records = samples.get(cls, [])
    if not records:
        axes[i, 1].axis('off')
        axes[i, 2].axis('off')
        continue

    rec = records[0]
    cls_idx = i

    try:
        img_path = TEST_DIR / rec['FileName']
        rgb_float, tensor = load_dicom_for_gradcam(img_path)

        cam_e = cam_eff(input_tensor=tensor, targets=[ClassifierOutputTarget(cls_idx)])
        cam_d = cam_dn( input_tensor=tensor, targets=[ClassifierOutputTarget(cls_idx)])

        overlay_e = show_cam_on_image(rgb_float, cam_e[0], use_rgb=True)
        overlay_d = show_cam_on_image(rgb_float, cam_d[0], use_rgb=True)

        prob_e = get_prob(eff_model, tensor, cls_idx)
        prob_d = get_prob(dn_model,  tensor, cls_idx)

        color_e = 'green' if prob_e >= 0.5 else 'red'
        color_d = 'green' if prob_d >= 0.5 else 'red'

        axes[i, 1].imshow(overlay_e)
        axes[i, 1].set_title(f'P = {prob_e:.3f}', fontsize=9, color=color_e)
        axes[i, 2].imshow(overlay_d)
        axes[i, 2].set_title(f'P = {prob_d:.3f}', fontsize=9, color=color_d)
    except Exception as e:
        for col in [1, 2]:
            axes[i, col].text(0.5, 0.5, f'Error:\n{str(e)[:50]}',
                              ha='center', va='center', fontsize=7, transform=axes[i, col].transAxes, color='red')

    axes[i, 1].axis('off')
    axes[i, 2].axis('off')

plt.tight_layout()
path_cmp = NB_DIR / 'ta004g_gradcam_comparacion.png'
plt.savefig(str(path_cmp), dpi=100, bbox_inches='tight')
wandb.log({'GradCAM/Comparacion_EFF_vs_DN': wandb.Image(str(path_cmp))})
print(f'Comparación guardada: {path_cmp.name}')
plt.show()

Comparación guardada: ta004g_gradcam_comparacion.png


C:\Users\Pcuser\AppData\Local\Temp\ipykernel_14936\1681778647.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Cierre W&B + resumen

In [14]:
wandb.summary.update({
    'n_classes':          NUM_CLASSES,
    'n_samples_per_class': N_SAMPLES,
    'eff_target_layer':   'features[-1] (Conv2dNormActivation)',
    'dn_target_layer':    'features.norm5 (BatchNorm2d)',
    'outputs': [
        'ta004g_gradcam_efficientnet.png',
        'ta004g_gradcam_densenet.png',
        'ta004g_gradcam_comparacion.png'
    ]
})

wandb.finish()

print('=== TA-004G Completado ===')
print(f'Archivos generados:')
for f in ['ta004g_gradcam_efficientnet.png', 'ta004g_gradcam_densenet.png', 'ta004g_gradcam_comparacion.png']:
    p = NB_DIR / f
    exists = '✓' if p.exists() else '✗'
    print(f'  {exists} {f}')

print('\nSiguiente: TA-005F — Threshold optimization + evaluación final del modelo ganador')

dn_target_layer,features.norm5 (Batc...
eff_target_layer,features[-1] (Conv2d...
n_classes,5
n_samples_per_class,3


=== TA-004G Completado ===
Archivos generados:
  ✓ ta004g_gradcam_efficientnet.png
  ✓ ta004g_gradcam_densenet.png
  ✓ ta004g_gradcam_comparacion.png

Siguiente: TA-005F — Threshold optimization + evaluación final del modelo ganador


## Notas técnicas

**Capas target usadas:**
- EfficientNet-B0: `model.features[-1]` — última capa Conv2dNormActivation antes del pooling global
- DenseNet-121: `model.features.norm5` — BatchNorm final después del denseblock4 (con `memory_efficient=False` para compatibilidad de gradientes)

**Interpretación del mapa de calor:**
- Rojo/naranja = zonas de mayor activación para esa clase
- Azul = zonas de menor relevancia
- P ≥ 0.5 = el modelo predice positivo para esa clase (verde), P < 0.5 = negativo (rojo)